In [1]:
Output = c("/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1014. Nanoparticles Respiratory Tract/P1014.3. Analyses/P1014.3.1. Group Distribution Analysis/Output")
cur_date = "021326"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(rlang)
library(limma)

# reading in files
protein_df = data.frame(read_excel("Input/Protein_Data_020926.xlsx", sheet = 2))[,4:13]
protein_df$Dose = as.numeric(protein_df$Dose)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘rlang’


The following objects are masked from ‘package:purrr’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice




In [2]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03001,Q07011,4-1BB,2.002402,1.586117
2,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03541,Q15109,AGER,3.425807,2.145941
3,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.178579,1.668382
4,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL04271,P31749,AKT1,2.214532,1.684609
5,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03101,Q15389,Angiopoietin-1,2.611234,1.852492
6,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL09731,P05089,ARG1,3.369304,2.127404


In [3]:
# only interested in centrifuged samples
protein_df = protein_df %>%
    filter(Separation == 'C')

we are interested in comparing NP1_Rhodamine B and NP2 with Vehicle control. For NP1 and NP1_Rhodamine B, two doses are available. 
Let's answer this with a 

Now on the data analysis side, we are interested in dose-dependent effect of NP1 and NP1_Rhodamine B. A comparison between NP1 and NP1_Rhodamine B will also be informative. Ultimately, the effect of NP2 and NP2_Rhodamine B (with only n=1) at the higher dose will be very informative. Comparison between NP1 and NP2 at the higher dose will be informative too.


## Is there a statistically significant difference in protein expression in respiratory tract cells based on treatment? How does this expression change with dose?

We'll use linear modeling to compare, NP1 (at both doses), NP1-Rhodamine B (at both doses) and NP2 to controls.

In [4]:
# converting to a wide format to run limma
wide_protein_matrix = protein_df %>%
    select(Sample_ID, UniProt_ID, Conc_pslog2) %>%
    pivot_wider(names_from = Sample_ID, values_from = Conc_pslog2) %>%
    column_to_rownames("UniProt_ID") %>%
    as.matrix()

head(wide_protein_matrix)

,Control_2_NA,NP1_3_0.1,NP1-Rhodamine B_5_0.1,NP1_6_0.05,NP1-Rhodamine B_8_0.05,Control_10_NA,NP1_11_0.1,NP1-Rhodamine B_13_0.1,NP1_14_0.05,NP2_Rhodamine B_15_0.1,NP1-Rhodamine B_16_0.05,Control_17_NA,NP2_18_0.1,NP1_20_0.1,NP1-Rhodamine B_21_0.1,NP2_22_0.1,NP1_24_0.05,NP1-Rhodamine B_25_0.05,NP2_26_0.1
Q07011,1.585350,1.569602,1.569492,1.572613,1.576097,1.583870,1.605838,1.582235,1.554672,1.554169,1.582363,1.575114,1.584561,1.592782,1.568494,1.573283,1.576927,1.566238,1.580319
Q15109,2.135693,2.138565,2.141932,2.143017,2.133122,2.073869,2.129192,2.129461,2.148347,2.137005,2.125730,2.138415,2.145104,2.115151,2.133203,2.128770,2.139980,2.125594,2.136470
Q9UNG2,1.642756,1.654883,1.636094,1.646805,1.645349,1.674437,1.661696,1.657174,1.650657,1.654289,1.649807,1.629017,1.652983,1.664520,1.664310,1.658886,1.660484,1.642435,1.642868
P31749,1.670583,1.680710,1.675403,1.663189,1.665997,1.663760,1.660594,1.666499,1.667402,1.663113,1.683948,1.649666,1.686324,1.671090,1.677939,1.693824,1.684437,1.676334,1.684229
Q15389,1.896098,1.773931,1.819644,1.803776,1.843614,1.877910,1.776046,1.818144,1.791555,1.709945,1.860328,1.851314,1.684969,1.783841,1.817430,1.677256,1.810487,1.823491,1.655942
P05089,2.120742,2.121573,2.120719,2.109544,2.123041,2.113760,2.128535,2.126708,2.149856,2.117535,2.133877,2.121512,2.109153,2.133167,2.131711,2.129999,2.132945,2.111452,2.146192


In [5]:
# creating a sample info df
sample_info_df = protein_df[,2:4] %>%
    unique() %>%
    # ensuring the sample ids are in the same order as the previous df
    arrange(match(Sample_ID, colnames(wide_protein_matrix)))

head(sample_info_df)

,Treatment,Sample_ID,Dose
,<chr>,<chr>,<dbl>
1,Control,Control_2_NA,0.00
2,NP1,NP1_3_0.1,0.10
3,NP1-Rhodamine B,NP1-Rhodamine B_5_0.1,0.10
4,NP1,NP1_6_0.05,0.05
5,NP1-Rhodamine B,NP1-Rhodamine B_8_0.05,0.05
6,Control,Control_10_NA,0.00


Here we want to see how dose affects treatment's effect on protein expression, hence the interaction. ie. Treatment + Dose

If we wanted to control for dose, it would help us determine treatment's effect *regardless* of dose. ie. Treatment * Dose

Can't run an interaction for a dependent variable with only one dose, so to keep it consistent across all treatments I'll just stratify to see how dose modifies protein expression too. 

In [20]:
# running limma
# comparing each tx group to the control while controlling for dose
design = model.matrix(~ Treatment, data = sample_info_df)
# MAY NOT WANT TO CONTROL FOR DOSE AND INSTEAD STRATIFY
linear_fit = lmFit(wide_protein_matrix, design)
limma_fit = eBayes(linear_fit)

treatments = colnames(design)[2:5]
limma_df = data.frame()
for (i in 1:length(treatments)){

    limma_results = topTable(limma_fit, coef = treatments[i], number = Inf) %>%
        rownames_to_column("UniProt_ID") %>%
        mutate(Treatment = treatments[i]) %>%
        # BH adjusted p value
        filter(adj.P.Val < 0.05)

    limma_df = rbind(limma_df, limma_results)

    }

# cleaning up the df
limma_df = limma_df[,c(1,2,5,6,8)] %>%
    mutate(Treatment = ifelse(grepl("Treatment", Treatment), 
                    unlist(strsplit(Treatment, split = "Treatment"))[2], Treatment))
colnames(limma_df)[3:4] = c("P Value", "P Adj")

# adding in meta data
limma_df = inner_join(limma_df, unique(protein_df[,6:8]))[,c(6,1,7,5,2:4)]
head(limma_df)

Joining with `by = join_by(UniProt_ID)`


,ELISA_ID,UniProt_ID,Protein_Name,Treatment,logFC,P Value,P Adj
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,nEL03731,P10909,CLU,NP1,-0.39103849,9.888882e-11,2.768887e-08
2,nEL04021,O00300,TNFRSF11B,NP1,-0.38900977,1.191029e-07,1.174437e-05
3,nEL00061,P10147,CCL3,NP1,0.11212664,1.258325e-07,1.174437e-05
4,nEL08971,Q99538,LGMN,NP1,-0.07179324,2.016804e-07,1.411763e-05
5,nEL03101,Q15389,Angiopoietin-1,NP1,-0.08516778,4.472279e-07,2.504476e-05
6,nEL01261,P49767,VEGF-C,NP1,-0.06259984,9.118955e-07,4.255512e-05


In [22]:
limma_df %>%
    filter(`P Adj` < 0.05)

ELISA_ID,UniProt_ID,Protein_Name,Treatment,logFC,P Value,P Adj
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
nEL03731,P10909,CLU,NP1,-0.39103849,9.888882e-11,2.768887e-08
nEL04021,O00300,TNFRSF11B,NP1,-0.38900977,1.191029e-07,1.174437e-05
nEL00061,P10147,CCL3,NP1,0.11212664,1.258325e-07,1.174437e-05
nEL08971,Q99538,LGMN,NP1,-0.07179324,2.016804e-07,1.411763e-05
nEL03101,Q15389,Angiopoietin-1,NP1,-0.08516778,4.472279e-07,2.504476e-05
nEL01261,P49767,VEGF-C,NP1,-0.06259984,9.118955e-07,4.255512e-05
nEL01981,Q99988,GDF-15 (MIC-1),NP1,0.06496027,1.185311e-06,4.741242e-05
nEL02901,P27930,IL-1 R2,NP1,0.06361153,6.223855e-06,2.178349e-04
nEL04541,P01135,TGFA,NP1,0.06811369,3.477226e-05,1.081804e-03
